# race90 보조 실행 화면

기본 작업은 로컬 터미널에서 분석·코딩·학습·추론으로 진행합니다. 이 Notebook은 선택적으로 쓰는 실행 화면입니다. 제출용 Notebook은 마지막에 원본 문제 파일을 보존하면서 자동 생성합니다.

90분 기준 70분 학습 종료 → 72분 회수 종료 → 84분 Notebook 완성 → 85분 ZIP 완성 → 마지막 5분 온라인 반영·업로드·접수 확인입니다. GPU는 CPU 기본 학습을 기다리지 않습니다.

대회 시작 즉시 터미널에서 `python -m race90 start-clock --output exam-clock.json`을 실행하세요. 문제를 읽고 다운로드한 시간까지 포함해야 합니다. 아래 실행 셀은 이 시계 파일을 사용합니다.

In [ ]:
from pathlib import Path
import sys, json, subprocess, time
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
CONFIG = ROOT / "config.yaml"
if not CONFIG.exists():
    raise FileNotFoundError("config.example.yaml을 config.yaml로 복사하고 실제 문제에 맞게 수정하세요.")
subprocess.run([sys.executable, "-m", "race90", "plan", str(CONFIG)], cwd=ROOT, check=True)

In [ ]:
# 같은 셀을 두 번 실행해 GPU를 중복 대여하지 않도록 실행 중인 프로세스를 확인합니다.
if "PROCESS" in globals() and PROCESS.poll() is None:
    raise RuntimeError("하네스가 이미 실행 중입니다. 아래 상태 셀을 확인하세요.")
CLOCK = ROOT / "exam-clock.json"
if not CLOCK.exists():
    raise FileNotFoundError("대회 시작 시 만든 exam-clock.json이 필요합니다. 공식 종료 시각을 기준으로 시계를 준비하세요.")
command = [sys.executable, "-m", "race90", "run", str(CONFIG), "--clock", str(CLOCK)]
RUN_LOG = ROOT / ("launch-" + str(int(time.time())) + ".log")
with RUN_LOG.open("w") as log:
    PROCESS = subprocess.Popen(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("하네스 PID:", PROCESS.pid, "로그:", RUN_LOG)

In [ ]:
# 반복 실행해서 상태를 확인하세요.
print("프로세스 종료 코드:", PROCESS.poll())
lines = RUN_LOG.read_text().splitlines()
print("\n".join(lines[-12:]))
RUN_DIR = None
for line in lines:
    try:
        event = json.loads(line)
    except json.JSONDecodeError:
        continue
    if event.get("event") == "started":
        RUN_DIR = Path(event["run_dir"])
        break
if RUN_DIR is not None:
    for filename in ["time-status.json", "status.json", "notebooks.json", "final.json"]:
        path = RUN_DIR / filename
        if path.exists():
            print(filename, path.read_text())

In [ ]:
# 완료 후 q*.solution.ipynb의 풀이 셀을 제공 Notebook에 반영하고 실행하세요.
# 제출 CSV는 아래 deliverables 폴더에 있습니다.
if RUN_DIR is not None and (RUN_DIR / "final.json").exists():
    result = json.loads((RUN_DIR / "final.json").read_text())
    print("파일 생성 완료:", result["complete"], "제출 준비:", result.get("submission_ready"))
    print("비상 기본값만 있는 문항:", result.get("fallback_tasks", []))
    for file in sorted((RUN_DIR / "deliverables").iterdir()):
        print(file)
    print("제출 ZIP:", RUN_DIR / "submission_pack.zip")
    print("온라인 제출 접수는 직접 확인해야 합니다.")
    print("문항별 대응 파일:", RUN_DIR / "deliverables/submission_checklist.json")

In [ ]:
# 조기에 제출본을 확보하고 싶을 때만 True로 바꾸어 실행하세요.
FINALIZE_NOW = False
if FINALIZE_NOW and RUN_DIR is not None:
    subprocess.run([sys.executable, "-m", "race90", "finalize", str(RUN_DIR)], cwd=ROOT, check=True)
